In [ ]:
import lightkurve
import matplotlib.pyplot as plot
import numpy
import triceratops.triceratops as tr
import json

%matplotlib inline
# plot.style.use('seaborn-v0_8')
parent_folder = 'CTOI Creation'

In [ ]:
data_paths = []

with open(f'{parent_folder}/queue_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

data_paths

In [ ]:
data_path = data_paths[0]
data_path

In [ ]:
with open(data_path, 'r') as file:
    data = json.load(file)

data

In [ ]:
star_id = data['id_num']
star_id

In [ ]:
triceratops = data['triceratops']
triceratops

In [ ]:
search_result = lightkurve.search_lightcurve(f'TIC {star_id}', mission = 'TESS')[:20]
search_result

In [ ]:
lc = search_result[12].download_all().stitch()
lc

In [ ]:
period = triceratops['period']
duration = triceratops['transit_duration'] / 24
transit_time = triceratops['transit_time']
depth = triceratops['transit_depth']
sectors = [12]
exptime = 200 / 86400

In [ ]:
# import pandas

# url = 'https://stev.oapd.inaf.it/tmp/output603559110852.dat'

# data = pandas.read_csv('trilegal.tbl', delim_whitespace = True, comment = '#')
# data

In [ ]:
# data.to_csv('trilegal.csv')

In [ ]:
lc.plot()

In [ ]:
# depth += -0.005 # add extra depth after seeing folded to improve value

In [ ]:
lc = lc.normalize()
flc = lc.fold(period = period, epoch_time = transit_time)
flc.errorbar()
plot.hlines([1 - depth], flc.time.min().value, flc.time.max().value)
plot.xlim(-duration * 2, duration * 2)
plot.show()

In [ ]:
mask = numpy.abs(flc.time) < duration * 2
mflc = flc[mask]
blc = mflc.bin(time_bin_size = (2 * duration * 2) / 200)
blc.errorbar()

In [ ]:
target = tr.target(ID = star_id, search_radius = 5, sectors = numpy.array(sectors), trilegal_fname = '')

In [ ]:
aps = target.get_spoc_apertures()
target.plot_field(sector = sectors[0], ap_pixels = aps[0] if aps != [] else None, ap_color = 'red')

In [ ]:
target.stars

In [ ]:
target.calc_depths(tdepth = depth, all_ap_pixels = aps)

In [ ]:
target.stars[target.stars.tdepth > 0]

In [ ]:
target.calc_probs(
    time = blc.time.value,                               # binned time data
    flux_0 = blc.flux.value,                             # binned flux data
    flux_err_0 = numpy.mean(blc.flux_err.value),            # standard deviation of (out-of-transit) flux data
    P_orb = period,                              # best-fit orbital period of planet candidate
    contrast_curve_file = None,        # contrast curve file to use (str)
    filt = 'TESS',                                # number of instances to simulate for each scenario (int, usually fine to leave alone)
    parallel = True,                               # whether or not to use parallel processing (bool)
    drop_scenario = ['DTP', 'DEB', 'DEBx2P', 'BTP', 'BEB', 'BEBx2P', 'NTP', 'NEB', 'NEB2xP'],                            # scenarios to ignore (e.g., "EB")
    verbose = 1,                                   # whether or not to print updates (0 or 1)
    flatpriors = False,                            # whether or not to assume flat planet priors (bool)
    exptime = exptime,                             # exposure time of TESS data (float, in days)
    molusc_file = None
)

In [ ]:
target.probs

In [ ]:
print('FPP = ', target.FPP) # eb on this star, etc.
print('NFPP = ', target.NFPP) # nearby contamination

In [ ]:
target.plot_fits(
    time = blc.time.value,                               # binned time data
    flux_0 = blc.flux.value,                             # binned flux data
    flux_err_0 = numpy.mean(blc.flux_err.value)
)